# Profiling das fontes para ingestão RAW

Evidência reproduzível para o contrato de ingestão da camada RAW.
O script não carrega dados no PostgreSQL e não executa transformações RAW -> CORE.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
from pathlib import Path

import pandas as pd

def find_repository_root(start: Path) -> Path:
    """Localiza a raiz do repositório sem depender do diretório de execução."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Não foi possível localizar a raiz do repositório a partir de "
        f"{start.resolve()}"
    )


REPOSITORY_ROOT = find_repository_root(Path.cwd())
DATA_DIR = REPOSITORY_ROOT / "data/raw"
OUTPUT_DIR = REPOSITORY_ROOT / "outputs/data-loading"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTRACT = {
    "olist_customers_dataset.csv": {
        "columns": [
            "customer_id", "customer_unique_id", "customer_zip_code_prefix",
            "customer_city", "customer_state",
        ],
        "keys": [["customer_id"]],
    },
    "olist_geolocation_dataset.csv": {
        "columns": [
            "geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
            "geolocation_city", "geolocation_state",
        ],
        "keys": [],
    },
    "olist_order_items_dataset.csv": {
        "columns": [
            "order_id", "order_item_id", "product_id", "seller_id",
            "shipping_limit_date", "price", "freight_value",
        ],
        "keys": [["order_id", "order_item_id"]],
    },
    "olist_order_payments_dataset.csv": {
        "columns": [
            "order_id", "payment_sequential", "payment_type",
            "payment_installments", "payment_value",
        ],
        "keys": [["order_id", "payment_sequential"]],
    },
    "olist_order_reviews_dataset.csv": {
        "columns": [
            "review_id", "order_id", "review_score", "review_comment_title",
            "review_comment_message", "review_creation_date",
            "review_answer_timestamp",
        ],
        "keys": [["review_id", "order_id"]],
    },
    "olist_orders_dataset.csv": {
        "columns": [
            "order_id", "customer_id", "order_status",
            "order_purchase_timestamp", "order_approved_at",
            "order_delivered_carrier_date", "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ],
        "keys": [["order_id"], ["customer_id"]],
    },
    "olist_products_dataset.csv": {
        "columns": [
            "product_id", "product_category_name", "product_name_lenght",
            "product_description_lenght", "product_photos_qty",
            "product_weight_g", "product_length_cm", "product_height_cm",
            "product_width_cm",
        ],
        "keys": [["product_id"]],
    },
    "olist_sellers_dataset.csv": {
        "columns": [
            "seller_id", "seller_zip_code_prefix", "seller_city", "seller_state",
        ],
        "keys": [["seller_id"]],
    },
    "product_category_name_translation.csv": {
        "columns": ["product_category_name", "product_category_name_english"],
        "keys": [["product_category_name"]],
    },
}

In [2]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def detect_encoding(path: Path) -> tuple[str, bool]:
    prefix = path.read_bytes()[:3]
    has_bom = prefix == b"\xef\xbb\xbf"
    # utf-8-sig também lê UTF-8 comum e remove BOM do cabeçalho quando presente.
    path.read_text(encoding="utf-8-sig")
    return ("utf-8-sig" if has_bom else "utf-8"), has_bom


def detect_dialect(path: Path, encoding: str) -> dict[str, object]:
    with path.open("r", encoding=encoding, newline="") as stream:
        sample = stream.read(65536)
    dialect = csv.Sniffer().sniff(sample, delimiters=",;\t|")
    return {
        "delimiter": dialect.delimiter,
        "quotechar": dialect.quotechar,
    }


def load_source(path: Path, encoding: str) -> pd.DataFrame:
    # dtype=str evita inferência de tipos.
    # keep_default_na=False impede que strings como "NA" sejam reinterpretadas.
    # na_values=[""] converte apenas campos vazios em ausência.
    return pd.read_csv(
        path,
        dtype=str,
        encoding=encoding,
        keep_default_na=False,
        na_values=[""],
    )


def key_profile(df: pd.DataFrame, keys: list[list[str]]) -> list[dict[str, object]]:
    result = []
    for cols in keys:
        duplicated = int(df.duplicated(subset=cols, keep=False).sum())
        distinct = int(df[cols].drop_duplicates().shape[0])
        result.append(
            {
                "columns": cols,
                "distinct_combinations": distinct,
                "duplicated_rows_by_key": duplicated,
                "is_unique": duplicated == 0,
            }
        )
    return result


def profile_file(filename: str, spec: dict[str, object]) -> dict[str, object]:
    path = DATA_DIR / filename
    if not path.exists():
        return {"file": filename, "present": False}

    encoding, has_bom = detect_encoding(path)
    dialect = detect_dialect(path, encoding)
    df = load_source(path, encoding)

    expected = list(spec["columns"])
    actual = list(df.columns)

    return {
        "file": filename,
        "present": True,
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
        "encoding": encoding,
        "has_utf8_bom": has_bom,
        **dialect,
        "rows": int(len(df)),
        "columns_count": int(len(df.columns)),
        "columns": actual,
        "has_header": actual == expected,
        "columns_match_contract": actual == expected,
        "nulls": {c: int(v) for c, v in df.isna().sum().items()},
        "distinct": {c: int(df[c].nunique(dropna=True)) for c in df.columns},
        "fully_duplicated_rows": int(df.duplicated(keep=False).sum()),
        "duplicate_rows_beyond_first": int(df.duplicated().sum()),
        "key_profiles": key_profile(df, list(spec["keys"])),
    }

In [3]:
profiles = [profile_file(filename, spec) for filename, spec in CONTRACT.items()]

missing = [p["file"] for p in profiles if not p["present"]]
structural_errors = [
    p["file"]
    for p in profiles
    if p["present"]
    and (
        not p["columns_match_contract"]
        or p["delimiter"] != ","
        or not p["has_header"]
    )
]

summary = {
    "files_expected": len(CONTRACT),
    "files_present": len(CONTRACT) - len(missing),
    "missing_files": missing,
    "structural_errors": structural_errors,
    "total_rows": sum(p.get("rows", 0) for p in profiles),
    "files": profiles,
}

OUTPUT_DIR.joinpath("raw_source_profile.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

pd.DataFrame(
    [
        {
            "arquivo": p["file"],
            "presente": p["present"],
            "linhas": p.get("rows"),
            "colunas": p.get("columns_count"),
            "tamanho_bytes": p.get("size_bytes"),
            "encoding": p.get("encoding"),
            "bom_utf8": p.get("has_utf8_bom"),
            "delimitador": p.get("delimiter"),
            "cabecalho": p.get("has_header"),
            "estrutura_valida": p.get("columns_match_contract"),
            "duplicadas_alem_primeira": p.get("duplicate_rows_beyond_first"),
            "sha256": p.get("sha256"),
        }
        for p in profiles
    ]
).to_csv(OUTPUT_DIR / "raw_source_inventory.csv", index=False)

In [4]:
print(f"Arquivos esperados: {summary['files_expected']}")
print(f"Arquivos presentes: {summary['files_present']}")
print(f"Total de registros: {summary['total_rows']:,}".replace(",", "."))
print(f"Arquivos ausentes: {missing or 'nenhum'}")
print(f"Erros estruturais: {structural_errors or 'nenhum'}")

if missing or structural_errors:
    raise SystemExit("PROFILING DAS FONTES RAW: REPROVADO")

print("PROFILING DAS FONTES RAW: APROVADO")

Arquivos esperados: 9
Arquivos presentes: 9
Total de registros: 1.550.922
Arquivos ausentes: nenhum
Erros estruturais: nenhum
PROFILING DAS FONTES RAW: APROVADO
